# Predicting Late Deliveries with Olist

Compare delivery-risk rankings under a 10% review budget.

In [2]:
import pandas as pd

orders = pd.read_csv(
    "olist_orders_dataset.csv",
    parse_dates=["order_delivered_customer_date", "order_estimated_delivery_date"],
)

print(f"Bestellungen insgesamt: {len(orders):,}")
orders.head()

Bestellungen insgesamt: 99,441


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


## Define late delivery

In [3]:
delivered = orders.loc[orders["order_status"].eq("delivered")].dropna(
    subset=["order_delivered_customer_date", "order_estimated_delivery_date"]
).copy()

actual_day = delivered["order_delivered_customer_date"].dt.normalize()
estimated_day = delivered["order_estimated_delivery_date"].dt.normalize()

delivered["is_late"] = actual_day > estimated_day

print(f"Auswertbare Bestellungen: {len(delivered):,}")
print(f"Davon verspätet: {delivered['is_late'].sum():,}")
print(f"Anteil verspätet: {delivered['is_late'].mean():.2%}")

Auswertbare Bestellungen: 96,470
Davon verspätet: 6,534
Anteil verspätet: 6.77%


## Explore promised delivery windows

In [5]:
purchase_day = pd.to_datetime(
    delivered["order_purchase_timestamp"]
).dt.normalize()

promised_day = delivered["order_estimated_delivery_date"].dt.normalize()
delivered["promised_days"] = (promised_day - purchase_day).dt.days

valid = delivered.loc[delivered["promised_days"].ge(0)].copy()

valid["delivery_window"] = pd.cut(
    valid["promised_days"],
    bins=[-1, 7, 14, 21, float("inf")],
    labels=["0–7 Tage", "8–14 Tage", "15–21 Tage", "22+ Tage"],
)

by_window = valid.groupby("delivery_window", observed=True).agg(
    bestellungen=("is_late", "size"),
    verspaetet=("is_late", "sum"),
    verspaetungsquote=("is_late", "mean"),
)

by_window["verspaetungsquote"] = by_window["verspaetungsquote"].map(
    "{:.2%}".format
)

by_window

,bestellungen,verspaetet,verspaetungsquote
delivery_window,,,
0–7 Tage,1260,213,16.90%
8–14 Tage,10742,532,4.95%
15–21 Tage,23304,1730,7.42%
22+ Tage,61164,4059,6.64%


## Split orders by purchase date

In [6]:
model_data = valid.assign(
    purchase_at=pd.to_datetime(valid["order_purchase_timestamp"])
).sort_values("purchase_at")

cutoff_index = int(len(model_data) * 0.8)
cutoff = model_data["purchase_at"].iloc[cutoff_index]

train = model_data.loc[
    (model_data["purchase_at"] < cutoff)
    & (model_data["order_delivered_customer_date"] < cutoff)
].copy()

test = model_data.loc[model_data["purchase_at"] >= cutoff].copy()

print(f"Stichtag: {cutoff:%d.%m.%Y}")
print(f"Training: {len(train):,} Bestellungen, {train['is_late'].mean():.2%} verspätet")
print(f"Test: {len(test):,} Bestellungen, {test['is_late'].mean():.2%} verspätet")
print(f"Test-Baseline – immer pünktlich: {1 - test['is_late'].mean():.2%} Accuracy")

Stichtag: 26.05.2018
Training: 75,128 Bestellungen, 7.11% verspätet
Test: 19,294 Bestellungen, 3.49% verspätet
Test-Baseline – immer pünktlich: 96.51% Accuracy


## Train a single-feature baseline

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix

features = ["promised_days"]

model = LogisticRegression()
model.fit(train[features], train["is_late"])

late_probability = model.predict_proba(test[features])[:, 1]
predicted_late = late_probability >= 0.5

tn, fp, fn, tp = confusion_matrix(
    test["is_late"], predicted_late, labels=[False, True]
).ravel()

print(f"Average Precision: {average_precision_score(test['is_late'], late_probability):.4f}")
print(f"Referenzwert für Average Precision: {test['is_late'].mean():.4f}")
print(f"Erkannte Verspätungen: {tp:,} von {tp + fn:,}")
print(f"Fehlalarme: {fp:,}")

Average Precision: 0.1203
Referenzwert für Average Precision: 0.0349
Erkannte Verspätungen: 0 von 674
Fehlalarme: 0


## Evaluate the 10% review budget

In [9]:
review = test[["order_id", "is_late"]].copy()
review["late_probability"] = late_probability

review_count = int(len(review) * 0.10)

selected = (
    review.sort_values(
        ["late_probability", "order_id"],
        ascending=[False, True],
    )
    .head(review_count)
)

found = selected["is_late"].sum()
total_late = review["is_late"].sum()

print(f"Zur Prüfung markiert: {len(selected):,}")
print(f"Erkannte Verspätungen: {found:,} von {total_late:,}")
print(f"Precision: {found / len(selected):.2%}")
print(f"Recall: {found / total_late:.2%}")

Zur Prüfung markiert: 1,929
Erkannte Verspätungen: 279 von 674
Precision: 14.46%
Recall: 41.39%


## Aggregate and join basket features

In [10]:
items = pd.read_csv("olist_order_items_dataset.csv")

order_features = items.groupby("order_id", as_index=False).agg(
    item_count=("order_item_id", "size"),
    product_value=("price", "sum"),
    freight_value=("freight_value", "sum"),
)

orders_with_items = model_data.merge(
    order_features,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print(f"Bestellungen vor dem Join: {len(model_data):,}")
print(f"Bestellungen nach dem Join: {len(orders_with_items):,}")
print(f"Ohne Artikeldaten: {orders_with_items['_merge'].eq('left_only').sum():,}")

orders_with_items[
    ["order_id", "promised_days", "item_count", "product_value", "freight_value"]
].head()

Bestellungen vor dem Join: 96,470
Bestellungen nach dem Join: 96,470
Ohne Artikeldaten: 0


,order_id,promised_days,item_count,product_value,freight_value
0,bfbd0f9bdef84302105ad712db648a6c,19,3,134.97,8.49
1,3b697a20d9e427646d92567910af6d57,24,1,29.90,15.56
2,be5bc2f0da14d8071e2d45451ad119d9,35,1,21.90,17.19
3,a41c8759fbe7aab36ea07e038b2d4465,57,1,36.49,17.24
4,d207cc272675637bfed0062edffd0818,51,1,119.90,13.56


## Compare logistic regression variants

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

development = orders_with_items.loc[
    (orders_with_items["purchase_at"] < cutoff)
    & (orders_with_items["order_delivered_customer_date"] < cutoff)
].sort_values("purchase_at")

validation_cutoff = development["purchase_at"].iloc[int(len(development) * 0.8)]

fit_data = development.loc[
    (development["purchase_at"] < validation_cutoff)
    & (development["order_delivered_customer_date"] < validation_cutoff)
]
validation = development.loc[development["purchase_at"] >= validation_cutoff]

feature_sets = {
    "A: Nur Lieferdauer": ["promised_days"],
    "B: Mit Artikeldaten": [
        "promised_days", "item_count", "product_value", "freight_value"
    ],
}

results = []

for name, features in feature_sets.items():
    pipeline = make_pipeline(StandardScaler(), LogisticRegression())
    pipeline.fit(fit_data[features], fit_data["is_late"])
    scores = pipeline.predict_proba(validation[features])[:, 1]

    ranked = validation[["order_id", "is_late"]].assign(score=scores)
    selected = ranked.sort_values(
        ["score", "order_id"], ascending=[False, True]
    ).head(int(len(validation) * 0.10))

    found = selected["is_late"].sum()
    results.append({
        "Modell": name,
        "Average Precision": average_precision_score(validation["is_late"], scores),
        "Erkannte Verspätungen": found,
        "Precision bei Top 10 %": found / len(selected),
        "Recall bei Top 10 %": found / validation["is_late"].sum(),
    })

print(f"Validierung ab: {validation_cutoff:%d.%m.%Y}")
print(f"Verspätungsquote in der Validierung: {validation['is_late'].mean():.2%}")
pd.DataFrame(results).round(4)


Validierung ab: 13.03.2018
Verspätungsquote in der Validierung: 5.58%


,Modell,Average Precision,Erkannte Verspätungen,Precision bei Top 10 %,Recall bei Top 10 %
0,A: Nur Lieferdauer,0.0501,47,0.0313,0.0561
1,B: Mit Artikeldaten,0.0779,138,0.0919,0.1647


## Evaluate gradient boosting

In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier

features = feature_sets["B: Mit Artikeldaten"]

boosting = HistGradientBoostingClassifier(
    max_iter=100,
    max_leaf_nodes=15,
    early_stopping=False,
    random_state=42,
)
boosting.fit(fit_data[features], fit_data["is_late"])
scores = boosting.predict_proba(validation[features])[:, 1]

ranked = validation[["order_id", "is_late"]].assign(score=scores)
selected = ranked.sort_values(
    ["score", "order_id"], ascending=[False, True]
).head(int(len(validation) * 0.10))

found = selected["is_late"].sum()

print(f"Average Precision: {average_precision_score(validation['is_late'], scores):.4f}")
print(f"Erkannte Verspätungen: {found:,}")
print(f"Precision bei Top 10 %: {found / len(selected):.2%}")
print(f"Recall bei Top 10 %: {found / validation['is_late'].sum():.2%}")

Average Precision: 0.0845
Erkannte Verspätungen: 144
Precision bei Top 10 %: 9.59%
Recall bei Top 10 %: 17.18%


## Compare models on later orders

In [13]:
from sklearn.base import clone

features = feature_sets["B: Mit Artikeldaten"]
later_orders = orders_with_items.loc[orders_with_items["purchase_at"] >= cutoff]

models = {
    "Logistische Regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "Gradient Boosting": clone(boosting),
}

results = []

for name, estimator in models.items():
    estimator.fit(development[features], development["is_late"])
    scores = estimator.predict_proba(later_orders[features])[:, 1]

    ranked = later_orders[["order_id", "is_late"]].assign(score=scores)
    selected = ranked.sort_values(
        ["score", "order_id"], ascending=[False, True]
    ).head(int(len(ranked) * 0.10))

    found = selected["is_late"].sum()
    results.append({
        "Modell": name,
        "Average Precision": average_precision_score(later_orders["is_late"], scores),
        "Geprüfte Bestellungen": len(selected),
        "Erkannte Verspätungen": found,
        "Precision bei Top 10 %": f"{found / len(selected):.2%}",
        "Recall bei Top 10 %": f"{found / ranked['is_late'].sum():.2%}",
    })

pd.DataFrame(results).round(4)

,Modell,Average Precision,Geprüfte Bestellungen,Erkannte Verspätungen,Precision bei Top 10 %,Recall bei Top 10 %
0,Logistische Regression,0.062,1929,102,5.29%,15.13%
1,Gradient Boosting,0.036,1929,81,4.20%,12.02%


## Compare delivery windows across periods

In [15]:
periods = pd.concat([
    validation.assign(period="Validierung"),
    later_orders.assign(period="Späterer Zeitraum"),
])

periods["delivery_window"] = pd.cut(
    periods["promised_days"],
    bins=[-1, 7, 14, 21, float("inf")],
    labels=["0–7 Tage", "8–14 Tage", "15–21 Tage", "22+ Tage"],
)

by_period = periods.groupby(
    ["delivery_window", "period"], observed=True
).agg(
    bestellungen=("is_late", "size"),
    verspaetungsquote=("is_late", "mean"),
)

by_period["verspaetungsquote"] = by_period["verspaetungsquote"].map("{:.2%}".format)
by_period

bestellungen verspaetungsquote
delivery_window period                                           
0–7 Tage        Späterer Zeitraum          1260            16.90%
8–14 Tage       Späterer Zeitraum          4135             5.18%
                Validierung                2400             2.62%
15–21 Tage      Späterer Zeitraum          4637             2.61%
                Validierung                4377             6.33%
22+ Tage        Späterer Zeitraum          9262             1.36%
                Validierung                8249             6.04%

## Inspect short delivery windows

In [16]:
simple_ranking = (
    review.assign(promised_days=test["promised_days"].to_numpy())
    .sort_values(["late_probability", "order_id"], ascending=[False, True])
    .head(int(len(review) * 0.10))
)

short_window = simple_ranking["promised_days"].between(0, 7)

print(f"Markierte Bestellungen mit 0–7 Tagen: {short_window.sum():,}")
print(
    "Davon verspätet: "
    f"{simple_ranking.loc[short_window, 'is_late'].sum():,}"
)

Markierte Bestellungen mit 0–7 Tagen: 1,260
Davon verspätet: 213


## Compare with a sorting rule

In [17]:
rule_selected = (
    test.sort_values(["promised_days", "order_id"])
    .head(len(simple_ranking))
)

rule_found = rule_selected["is_late"].sum()

print(f"Erkannte Verspätungen mit der Regel: {rule_found:,}")
print(
    "Dieselben Bestellungen ausgewählt:",
    set(rule_selected["order_id"]) == set(simple_ranking["order_id"]),
)

Erkannte Verspätungen mit der Regel: 279
Dieselben Bestellungen ausgewählt: True
